In [1]:
from pathlib import Path
import ast
import random

import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT = Path.cwd()
TRAIN_DIR = ROOT / "Dataset"
SENTENCE_DIR = ROOT / "sentence_data2"

for directory in (TRAIN_DIR, SENTENCE_DIR):
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing folder: {directory}")

print("Project folder:", ROOT)

Project folder: /Users/adwaiy/Desktop/Sign-Language-Detection


In [2]:
def parse_frame(line):
    items = ast.literal_eval(line)
    features = []

    for item in items:
        if isinstance(item, str):
            item = ast.literal_eval(item)

        if isinstance(item, (tuple, list)) and len(item) == 3:
            features.extend(float(value) for value in item)
        elif isinstance(item, (int, float)) and item == 0:
            features.extend([0.0, 0.0, 0.0])
        else:
            raise ValueError(f"Unexpected frame entry: {item!r}")

    if len(features) != 36:
        raise ValueError(f"Expected 36 features, got {len(features)}")

    if not np.isfinite(features).all():
        raise ValueError("Recording contains non-finite values")

    return features


def load_recording(path):
    with path.open(encoding="utf-8") as file:
        frames = [
            parse_frame(line)
            for line in file
            if line.strip()
        ]

    recording = np.asarray(frames, dtype=np.float32)

    if recording.shape != (90, 36):
        raise ValueError(
            f"{path.name}: expected (90, 36), got {recording.shape}"
        )

    return recording


files = sorted(TRAIN_DIR.glob("*/*.txt"))

if not files:
    raise FileNotFoundError("No training .txt files found")

X = np.stack([load_recording(path) for path in files])
word_labels = [path.name.split("_")[0] for path in files]

encoder = LabelEncoder()
y = encoder.fit_transform(word_labels)
n_classes = len(encoder.classes_)

print("Dataset shape:", X.shape)
print("Classes:", n_classes)
print("Labels:", encoder.classes_)

Dataset shape: (378, 90, 36)
Classes: 42
Labels: ['A' 'AFTER' 'AM' 'ARE' 'B' 'BAD' 'BALL' 'BLUE' 'C' 'CALL' 'CARROM'
 'COLLEGE' 'ENGLISH' 'EXCUSE' 'FINE' 'FOOD' 'FRIEND' 'GOOD' 'HAPPY' 'HELP'
 'HI' 'HOW' 'I' 'MORNING' 'NAME' 'ORANGE' 'PHONE' 'PLACE' 'PLEASE' 'SORRY'
 'TALK' 'THANK' 'THINK' 'THRUSDAY' 'TIME' 'TOILET' 'TUESDAY' 'WHAT'
 'WHERE' 'WHICH' 'WORK' 'YOU']


In [3]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

X_fit_raw, X_valid_raw, y_fit, y_valid = train_test_split(
    X_train_raw,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train,
)

scaler = StandardScaler()
scaler.fit(X_fit_raw.reshape(-1, 36))


def scale_recordings(recordings):
    shape = recordings.shape
    scaled = scaler.transform(recordings.reshape(-1, 36))
    return scaled.reshape(shape).astype(np.float32)


X_train = scale_recordings(X_train_raw)
X_test = scale_recordings(X_test_raw)
X_fit = scale_recordings(X_fit_raw)
X_valid = scale_recordings(X_valid_raw)

print("SVM training:", X_train.shape)
print("LSTM training:", X_fit.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

SVM training: (264, 90, 36)
LSTM training: (211, 90, 36)
Validation: (53, 90, 36)
Test: (114, 90, 36)


In [4]:
svm = SVC()

svm.fit(
    X_train.reshape(len(X_train), -1),
    y_train,
)

svm_predictions = svm.predict(
    X_test.reshape(len(X_test), -1)
)

print("SVM test accuracy:",
      accuracy_score(y_test, svm_predictions))
print("SVM test macro-F1:",
      f1_score(y_test, svm_predictions, average="macro"))

SVM test accuracy: 0.9298245614035088
SVM test macro-F1: 0.9228458049886621


In [5]:
model = Sequential([
    Input(shape=(90, 36)),
    LSTM(120),
    Dropout(0.3),
    Dense(100, activation="relu"),
    Dense(n_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_fit,
    y_fit,
    validation_data=(X_valid, y_valid),
    epochs=50,
    batch_size=32,
    verbose=1,
)

lstm_predictions = np.argmax(
    model.predict(X_test, verbose=0),
    axis=1,
)

print("LSTM test accuracy:",
      accuracy_score(y_test, lstm_predictions))
print("LSTM test macro-F1:",
      f1_score(y_test, lstm_predictions, average="macro"))

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.0379 - loss: 3.7160 - val_accuracy: 0.3019 - val_loss: 3.4762
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2654 - loss: 3.4015 - val_accuracy: 0.4717 - val_loss: 3.2065
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4455 - loss: 3.0909 - val_accuracy: 0.4906 - val_loss: 2.9118
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5071 - loss: 2.7663 - val_accuracy: 0.6226 - val_loss: 2.5765
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6066 - loss: 2.4031 - val_accuracy: 0.7170 - val_loss: 2.1982
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6967 - loss: 2.0216 - val_accuracy: 0.7925 - val_loss: 1.8115
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7915 - loss: 1.6127 - val_accuracy: 0.8679 - val_loss: 1.4305
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8768 - loss: 1.2593 - val_accuracy: 0.8679 - val_loss: 1.0922


In [6]:
word_recordings = {
    path.stem: load_recording(path)
    for path in sorted(SENTENCE_DIR.glob("*.txt"))
}

sentences = [
    "HI",
    "I AM FINE",
    "WHAT TIME YOU WORK",
    "I WORK TUESDAY MORNING",
    "I WORK THRUSDAY MORNING",
    "FRIEND TALK FINE ENGLISH",
    "PLEASE CALL FRIEND THRUSDAY MORNING",
    "AM I BAD",
    "I AM GOOD",
    "I AM HAPPY",
    "YOU ARE GOOD FRIEND",
    "WHERE ARE YOU",
    "I AM BAD",
    "WHERE AM I",
    "GOOD MORNING FRIEND",
    "GOOD WORK",
    "THANK YOU",
    "YOU ARE COLLEGE FRIEND",
    "WHAT YOU WORK",
    "GOOD PLACE",
    "I AM SORRY",
    "I WORK MORNING",
    "I THINK YOU ARE HAPPY",
    "I THINK YOU ARE FINE",
    "WHERE YOU WORK",
    "HOW ARE YOU",
    "I WORK AFTER FOOD",
    "PLEASE HELP",
]

lstm_scores = []
svm_scores = []

for sentence in sentences:
    words = sentence.split()

    missing = [word for word in words if word not in word_recordings]
    if missing:
        raise ValueError(f"Missing word recordings: {missing}")

    recordings = np.stack([word_recordings[word] for word in words])
    recordings = scale_recordings(recordings)
    actual = encoder.transform(words)

    lstm_pred = np.argmax(
        model.predict(recordings, verbose=0),
        axis=1,
    )

    svm_pred = svm.predict(
        recordings.reshape(len(recordings), -1)
    )

    lstm_scores.append(accuracy_score(actual, lstm_pred))
    svm_scores.append(accuracy_score(actual, svm_pred))

    print("Input:     ", sentence)
    print("LSTM words:", " ".join(encoder.inverse_transform(lstm_pred)))
    print()

print("LSTM demo mean word accuracy:",
      np.mean(lstm_scores))
print("SVM demo mean word accuracy:",
      np.mean(svm_scores))

Input:      HI
LSTM words: HI

Input:      I AM FINE
LSTM words: I AM FINE

Input:      WHAT TIME YOU WORK
LSTM words: WHAT TIME YOU WORK

Input:      I WORK TUESDAY MORNING
LSTM words: I WORK TUESDAY MORNING

Input:      I WORK THRUSDAY MORNING
LSTM words: I WORK THRUSDAY MORNING

Input:      FRIEND TALK FINE ENGLISH
LSTM words: FRIEND TALK FINE ENGLISH

Input:      PLEASE CALL FRIEND THRUSDAY MORNING
LSTM words: PLEASE CALL FRIEND THRUSDAY MORNING

Input:      AM I BAD
LSTM words: AM I BAD

Input:      I AM GOOD
LSTM words: I AM GOOD

Input:      I AM HAPPY
LSTM words: I AM TOILET

Input:      YOU ARE GOOD FRIEND
LSTM words: YOU ARE GOOD FRIEND

Input:      WHERE ARE YOU
LSTM words: WHERE ARE YOU

Input:      I AM BAD
LSTM words: I AM BAD

Input:      WHERE AM I
LSTM words: WHERE AM I

Input:      GOOD MORNING FRIEND
LSTM words: GOOD MORNING FRIEND

Input:      GOOD WORK
LSTM words: GOOD WORK

Input:      THANK YOU
LSTM words: THANK YOU

Input:      YOU ARE COLLEGE FRIEND
LSTM words: